In [1]:
import pathlib
import sqlite3

In [2]:
path = pathlib.Path('data/stocklab.db')

connection = sqlite3.connect(path)
connection.row_factory = sqlite3.Row


In [3]:
# data = connection.execute(
#     "SELECT * FROM stockdaily WHERE id == ? ORDER BY date ASC",
#     ('2327',)
# ).fetchall()

data = connection.execute(
    "SELECT * FROM twsedaily ORDER BY date ASC",
).fetchall()

In [5]:
import pandas as pd

df = pd.DataFrame([{k:v for k,v in zip(r.keys(), r)} for r in data])
df['date'] = pd.to_datetime(df["date"])

print(df.columns)

exp = ['total_outstanding_shares','foreign_balance_volume']
con1 = df.drop(['date']+exp,axis=1).isna().all(axis=1)
con2 = ~df[exp].isna().any(axis=1)
df = df.loc[~(con1&con2)]
# 外資持股和在外流通股數 會在年假的前面兩天有資料 但其實已經沒有交易了 需踢除

Index(['date', 'total_outstanding_shares', 'open', 'high', 'low', 'close',
       'volume', 'turnover', 'trades', 'margin_buy_volume',
       'margin_sell_volume', 'margin_cash_repayment_volume',
       'margin_balance_volume', 'margin_buy_value', 'margin_sell_value',
       'margin_cash_repayment_value', 'margin_balance_value',
       'short_sell_volume', 'short_cover_volume',
       'short_stock_repayment_volume', 'short_balance_volume',
       'short_sell_value', 'sbl_sell_volume', 'sbl_return_volume',
       'sbl_adjustment_volume', 'sbl_ex_sell_change_volume',
       'sbl_balance_volume', 'sbl_sell_value', 'foreign_ex_dealer_buy_volume',
       'foreign_ex_dealer_sell_volume', 'foreign_dealer_buy_volume',
       'foreign_dealer_sell_volume', 'foreign_buy_volume',
       'foreign_sell_volume', 'foreign_balance_volume',
       'foreign_ex_dealer_buy_value', 'foreign_ex_dealer_sell_value',
       'foreign_dealer_buy_value', 'foreign_dealer_sell_value',
       'foreign_buy_value', 'fo

In [6]:
# vol_cols = df.columns.drop(['date','open','high','low','close'])
# print(len(vol_cols))

# import core.database.schema.tables as tables
# vol_cols = list(tables.TWSEDaily.f_margin_balance_volume.items.values())+list(tables.TWSEDaily.f_margin_balance_value.items.values())
# vol_cols = list(tables.TWSEDaily.f_short_balance_volume.items.values())+list(tables.TWSEDaily.f_sbl_balance_volume.items.values())
# vol_cols = vol_cols[-2:]

vol_cols = ['volume','sbl_balance_volume']


In [9]:
import plotly.graph_objects as go

# 1. Calculate specific grid positions (domains) for your subplots
num_subplots = 1 + len(vol_cols)
spacing = 0.04  # Blank padding gap between plot rows
available_height = 1.0 - (spacing * (num_subplots - 1))

# Allocate 60% of the canvas height to the main OHLC chart, split the rest evenly
ohlc_weight = 0.60
vol_weight = (1.0 - ohlc_weight) / len(vol_cols) if vol_cols else 0.40

# Generate coordinates for your separate rows
domains = []
current_bottom = 0.0
for i in range(len(vol_cols)):
    domains.append([current_bottom, current_bottom + (available_height * vol_weight)])
    current_bottom += (available_height * vol_weight) + spacing
domains.append([current_bottom, 1.0])  # Top plot gets the remainder up to 1.0

# Reverse so index matches row numbering from top to bottom
domains.reverse()

# 2. Initialize a blank Figure instead of using breakable subplots
fig = go.Figure()

# 3. Add the Candlestick chart to the top row domain
fig.add_trace(
    go.Candlestick(
        x=df["date"],
        open=df["open"],
        high=df["high"],
        low=df["low"],
        close=df["close"],
        name="OHLC",
        xaxis="x",         # Shared single tracking axis
        yaxis="y",         # Mapped to top row position
        # hoverinfo="none",
    )
)
fig.add_trace(
    go.Scatter(
        x=df["date"],
        y=df['turnover']/df['volume'],
        name='avg',
        xaxis="x",         # Shared single tracking axis
        yaxis="y",         # Mapped to top row position
    )
)

# 4. Add the volume bar charts to lower row domains
for i, c in enumerate(vol_cols):
    fig.add_trace(
        go.Bar(
            x=df["date"],
            y=df[str(c)],
            name=str(c),
            xaxis="x",                 # Linked to the exact same x axis for spikelines
            yaxis=f"y{i+2}",           # Distributed to separate Y variables
            # width=1000
        )
    )

# 5. Inject layout parameters and activate the spanning line engine
layout_kwargs = {
    "xaxis_rangeslider_visible": False,
    "width": 1200,
    "height": 800,
    "hovermode": "x",           # Joins labels under one crosshair coordinate
    "hoverlabel": dict(
        bgcolor="rgba(255, 255, 255, 0.1)",  # White background with 60% opacity
        bordercolor="rgba(0, 0, 0, 0.2)",     # Faint border line
        font=dict(color="black")              # Text color inside the box
    ),
    "xaxis": dict(
        showspikes=True,               # Turns on vertical crosshairs
        spikemode="across",            # Instructs line to span the entire screen height
        spikesnap="cursor",            # Pins line directly beneath your cursor
        spikethickness=0.5,
        spikedash="dash",
        spikecolor="gray",
    ),
    "yaxis": dict(
        showspikes=True,               # Turns on vertical crosshairs
        spikemode="across",            # Instructs line to span the entire screen height
        spikesnap="cursor",            # Pins line directly beneath your cursor
        spikethickness=0.5,
        spikedash="dash",
        spikecolor="gray",
        domain=domains[0]
    ),  # Apply the computed top layout space
}

# Bind individual layout domains to keep rows physically separated
for i in range(len(vol_cols)):
    layout_kwargs[f"yaxis{i+2}"] = layout_kwargs['yaxis'].copy()
    layout_kwargs[f"yaxis{i+2}"]['domain'] = domains[i+1]

fig.update_layout(**layout_kwargs)

# 6. Apply your rangebreaks directly to the single unified x-axis
fig.update_xaxes(
    rangebreaks=[
        dict(bounds=["sat", "mon"], pattern="day of week")
    ]
)

fig.show()
